# concat连接
实现数据的堆叠

## 1)Series与Series连接

In [3]:
import pandas as pd
s1 = pd.Series(["A", "B"], index=[1, 2],name='s1')
s2 = pd.Series(["D", "E"], index=[4, 5],name='s2')
s3 = pd.Series(["G", "H"], index=[7, 8],name='s3')

# 注意: 如果是Series,只有一个维度,按行进行堆叠的时候,名字没有影响
pd.concat([s1, s2, s3])

pd.concat([s1, s2, s3], axis=1)


,s1,s2,s3
1,A,NaN,NaN
2,B,NaN,NaN
4,NaN,D,NaN
5,NaN,E,NaN
7,NaN,NaN,G
8,NaN,NaN,H


## 2)DataFrame与Series连接

In [5]:
df1 = pd.DataFrame(data={"a": [1, 2], "b": [4, 5]}, index=[1, 2])
s1 = pd.Series(data=[7, 10], index=[1, 2], name="a")

pd.concat([df1,s1]) # 按行连接,会先去找列名一致
pd.concat([df1,s1], axis=1)     # 按列连接,会先去找行名一致

,a,b,a
1,1,4,7
2,2,5,10


## 3)DataFrame与DataFrame连接

In [13]:
df1 = pd.DataFrame(data={"a": [1, 2], "b": [4, 5]}, index=[1, 2])
df2 = pd.DataFrame(data={"a": [7, 8], "b": [10, 11]}, index=[1, 2])

pd.concat([df1,df2])    # 按行连接,会先去找列名一致
pd.concat([df1,df2], axis=1)    # 按列连接,会先去找行名一致


# ignore_index=True 重置行标签
pd.concat([df1,df2],axis=0,ignore_index=True)

,index,a,b
0,1,1,4
1,2,2,5
2,1,7,10
3,2,8,11


In [ ]:
## 5)类似join的连接

In [16]:
df1 = pd.DataFrame(data={"a": [1, 2], "b": [4, 5]}, index=[1, 2])
df2 = pd.DataFrame(data={"b": [7, 8], "c": [10, 11]}, index=[2, 3])

pd.concat([df1,df2])
pd.concat([df1,df2],join="inner")    # 内连接

,b
1,4
2,5
2,7
3,8


# merge合并

## 1)数据连接的类型

In [26]:
# (1) 一对一连接
df1 = pd.DataFrame(
    {"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting", "Engineering", "Engineering", "HR"]}
)

df2 = pd.DataFrame({"employee": ["Lisa", "Bob", "Jake", "Sue"], "hire_date": [2004, 2008, 2012, 2014]})

# merge第一种使用方式
df1.merge(df2)
# merge第二种
pd.merge(df1,df2)

# 如果没有指定两个DataFrame的关联字段,默认情况,到两个DataFrame中找名字相同的字段进行关联

# （2）多对一连接
# 在需要连接的两个列中，有一列的值有重复。通过多对一连接获得的结果将会保留重复值。
df1 = pd.DataFrame(
    {"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting", "Engineering", "Engineering", "HR"]}
)

df2 = pd.DataFrame({"group": ["Accounting", "Engineering", "HR"], "supervisor": ["Carly", "Guido", "Steve"]})

pd.merge(df1,df2)

# （3）多对多连接
# 如果左右两个输入的共同列都包含重复值，那么合并的结果就是一种多对多连接。
df1 = pd.DataFrame(
    {"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting", "Engineering", "Engineering", "HR"]}
)
df2 = pd.DataFrame(
    {
        "group": ["Accounting", "Accounting", "Engineering", "Engineering", "HR", "HR"],
        "skills": ["math", "spreadsheets", "coding", "linux", "spreadsheets", "organization"],
    }
)
pd.merge(df1,df2)



,employee,group,skills
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,coding
3,Jake,Engineering,linux
4,Lisa,Engineering,coding
5,Lisa,Engineering,linux
6,Sue,HR,spreadsheets
7,Sue,HR,organization


## 设置合并的键与索引

In [37]:
# （1）通过on指定使用某个列连接，只能在有共同列名的时候使用
df1 = pd.DataFrame(
    {"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting", "Engineering", "Engineering", "HR"]}
)
df2 = pd.DataFrame({"employee": ["Lisa", "Bob", "Jake", "Sue"], "hire_date": [2004, 2008, 2012, 2014]})

pd.merge(df1,df2,on="employee")

# （2）两对象列名不同，通过left_on和right_on分别指定列名
df1 = pd.DataFrame(
    {"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting", "Engineering", "Engineering", "HR"]}
)
df2 = pd.DataFrame({"name": ["Bob", "Jake", "Lisa", "Sue"], "salary": [70000, 80000, 120000, 90000]})

pd.merge(df1,df2,left_on="employee",right_on="name")

# （3）通过left_index和right_index设置合并的索引
# 通过设置merge()中的left_index、right_index参数将索引设置为键来实现合并。
df1 = pd.DataFrame(
    {"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting", "Engineering", "Engineering", "HR"]}
)
df2 = pd.DataFrame({"employee": ["Lisa", "Bob", "Jake", "Sue"], "hire_date": [2004, 2008, 2012, 2014]})

df1.set_index('employee', inplace=True)
df2.set_index('employee', inplace=True)
pd.merge(df1,df2,left_index=True,right_index=True)


# join()        merge()==>按照索引进行关联的简化
df1 = pd.DataFrame({
    'key': ['A', 'B', 'C'],
    'value1': [1, 2, 3]
})

df2 = pd.DataFrame({
    'key': ['B', 'C', 'D'],
    'value2': [4, 5, 6]
})

df1.merge(df2)
pd.merge(df1,df2,left_index=True,right_index=True)
df1.join(df2,lsuffix='_left',rsuffix='_right')      # 按照索引进行数据合并。但要求没有重叠的列，或通过lsuffix、rsuffix指定重叠列的后缀。

,key_left,value1,key_right,value2
0,A,1,B,4
1,B,2,C,5
2,C,3,D,6
